In [ ]:
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from itertools import combinations

## Построение baseline

In [40]:
df_rez_base = df_rez_base[['id', 'resume_title', 'skills_list', 'experience_text']].copy()
df_vac_base = df_vac_base[['id', 'vacancy_name', 'skills_list', 'experience_years_min']].copy()

In [94]:
df_rez_base.shape

(11782, 4)

In [95]:
df_vac_base.shape

(571, 4)

### Формирование полуэталона - построение range на основе навыков совпадающих навыков и опыта работы

Для данной задачи был использован следующий метод:

Для расчета score по опыту работы
$$
exp\_score =
\begin{cases}
1, & \text{если } resume\_exp \ge vacancy\_exp \\
\dfrac{resume\_exp}{vacancy\_exp}, & \text{если } resume\_exp < vacancy\_exp
\end{cases}
$$

При расчете score по навыкам:
$$
skill\_score =
\frac{|resume\_skills \cap vacancy\_skills|}{|vacancy\_skills|}
$$

Итоговый score

$$
final\_score = 0.67 \cdot skill\_score + 0.33 \cdot exp\_score
$$


In [41]:
def skill_match_score(resume_skills, vacancy_skills):
    resume_set = set(resume_skills) if isinstance(resume_skills, list) else set()
    vacancy_set = set(vacancy_skills) if isinstance(vacancy_skills, list) else set()

    if len(vacancy_set) == 0:
        return 0.0, 0

    matched_count = len(resume_set & vacancy_set)
    skill_score = matched_count / len(vacancy_set)

    return skill_score, matched_count


def exp_match_score(resume_exp, vacancy_exp):
    if pd.isna(resume_exp):
        resume_exp = 0
    if pd.isna(vacancy_exp):
        vacancy_exp = 0

    if vacancy_exp == 0:
        return 1.0

    if resume_exp >= vacancy_exp:
        return 1.0

    return resume_exp / vacancy_exp


def final_match_score(resume_skills, vacancy_skills, resume_exp, vacancy_exp):
    skill_score, matched_count = skill_match_score(resume_skills, vacancy_skills)
    exp_score = exp_match_score(resume_exp, vacancy_exp)

    final_score = (2/3) * skill_score + (1/3) * exp_score

    return {
        'skill_score': skill_score,
        'matched_skills_count': matched_count,
        'exp_score': exp_score,
        'final_score': final_score
    }

In [42]:
def rank_vacancies_for_resume(
    resume_row,
    df_vac,
    resume_skills_col='skills_list',
    resume_exp_col='experience_text',
    vacancy_skills_col='skills_list',
    vacancy_exp_col='experience_years_min'
):
    results = []

    for _, vac_row in df_vac.iterrows():
        scores = final_match_score(
            resume_skills=resume_row[resume_skills_col],
            vacancy_skills=vac_row[vacancy_skills_col],
            resume_exp=resume_row[resume_exp_col],
            vacancy_exp=vac_row[vacancy_exp_col]
        )

        results.append({
            'resume_id': resume_row['id'],
            'resume_title': resume_row['resume_title'],
            'vacancy_id': vac_row['id'],
            'vacancy_name': vac_row['vacancy_name'],
            'vacancy_exp': vac_row[vacancy_exp_col],
            'skill_score': scores['skill_score'],
            'matched_skills_count': scores['matched_skills_count'],
            'exp_score': scores['exp_score'],
            'final_score': scores['final_score']
        })

    result_df = pd.DataFrame(results)

    result_df = result_df.sort_values(
        by=['final_score', 'matched_skills_count'],
        ascending=[False, False]
    ).reset_index(drop=True)

    result_df['rank'] = range(1, len(result_df) + 1)

    return result_df

In [43]:
all_matches = []

for _, resume_row in df_rez_base.iterrows():
    ranked = rank_vacancies_for_resume(
        resume_row=resume_row,
        df_vac=df_vac_base,
        resume_skills_col='skills_list',
        resume_exp_col='experience_text',
        vacancy_skills_col='skills_list',
        vacancy_exp_col='experience_years_min'
    )
    all_matches.append(ranked)

matches_df = pd.concat(all_matches, ignore_index=True)
matches_df.head()

,resume_id,resume_title,vacancy_id,vacancy_name,vacancy_exp,skill_score,matched_skills_count,exp_score,final_score,rank
0,27934,Data Scientist,4486,Junior Data Scientist,0.0,1.000000,1,1.0,1.000000,1
1,27934,Data Scientist,10404,Аналитик данных / Специалист по CRM,0.0,1.000000,1,1.0,1.000000,2
2,27934,Data Scientist,6769,Junior Data Engineer / Разработчик ETL,0.0,0.666667,2,1.0,0.777778,3
3,27934,Data Scientist,4676,Data Scientist,3.0,1.000000,3,0.0,0.666667,4
4,27934,Data Scientist,6183,Аналитик данных,1.0,1.000000,3,0.0,0.666667,5


In [44]:
matches_df.shape

(6727522, 10)

In [45]:
def select_top_matches(group, top_unique_scores=5):
    group = group.sort_values(
        by=["final_score", "matched_skills_count", "vacancy_exp"],
        ascending=[False, False, False]
    ).copy()

    top_scores = (
        group.loc[group["final_score"] > 0, "final_score"]
        .drop_duplicates()
        .head(top_unique_scores)
    )

    filtered = group[group["final_score"].isin(top_scores)].copy()

    filtered = filtered.sort_values(
        by=["final_score", "matched_skills_count", "vacancy_exp"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    filtered["rank"] = range(1, len(filtered) + 1)

    return filtered

In [46]:
result = (
    matches_df
    .groupby("resume_id", group_keys=False)
    .apply(select_top_matches)
    .reset_index(drop=True)
)

/var/folders/h7/k0zwpy4s6tz5mbdzhg164w_m0000gn/T/ipykernel_88702/559495581.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_top_matches)


In [47]:
cols_to_show = [
    'resume_id',
    'resume_title',
    'vacancy_id',
    'vacancy_name',
    'matched_skills_count',
    'skill_score',
    'exp_score',
    'final_score'
]

result[cols_to_show].head(30)

,resume_id,resume_title,vacancy_id,vacancy_name,matched_skills_count,skill_score,exp_score,final_score
0,2,Аналитик данных,9445,ML Engineer (LLM / RAG),1,1.000000,1.0,1.000000
1,2,Аналитик данных,4486,Junior Data Scientist,1,1.000000,1.0,1.000000
2,2,Аналитик данных,9083,Team Lead Data Analyst,4,0.800000,1.0,0.866667
3,2,Аналитик данных,4195,Data Engineer,3,0.750000,1.0,0.833333
4,2,Аналитик данных,5136,Junior Data Analyst (mobile games),3,0.750000,1.0,0.833333
5,2,Аналитик данных,7931,Аналитик данных,5,0.714286,1.0,0.809524
6,2,Аналитик данных,10143,Аналитик данных (Fraud),5,0.714286,1.0,0.809524
7,2,Аналитик данных,6507,Аналитик данных (senior),2,0.666667,1.0,0.777778
8,2,Аналитик данных,7958,Аналитик данных / Data analyst,2,0.666667,1.0,0.777778
9,2,Аналитик данных,8164,Аналитик данных Power BI,2,0.666667,1.0,0.777778


In [93]:
pseudo_counts = result.groupby("resume_id")["vacancy_id"].nunique()
pseudo_counts.describe()

count    11782.000000
mean        63.330080
std        107.564487
min          5.000000
25%         25.000000
50%         40.000000
75%         60.000000
max        571.000000
Name: vacancy_id, dtype: float64

### Построение baseline только на текстах вакансий

In [48]:
resume_texts = df_rez["wrapper_text"].tolist()
vacancy_texts = df_vac["wrapper_text"].tolist()

In [ ]:
all_texts = resume_texts + vacancy_texts

vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=2
)

all_matrix = vectorizer.fit_transform(all_texts)

resume_matrix = all_matrix[:len(resume_texts)]
vacancy_matrix = all_matrix[len(resume_texts):]

sim_matrix = cosine_similarity(resume_matrix, vacancy_matrix)

In [85]:
k = 5

top_k_indices = np.argsort(-sim_matrix, axis=1)[:, :k]
top_k_scores = np.take_along_axis(sim_matrix, top_k_indices, axis=1)

In [86]:
results = []

for i in range(len(df_rez_base)):
    for rank in range(k):
        vac_idx = top_k_indices[i, rank]
        score = top_k_scores[i, rank]

        results.append({
            "resume_id": df_rez_base.iloc[i]["id"],
            "resume_title": df_rez_base.iloc[i]["resume_title"],
            "vacancy_id": df_vac_base.iloc[vac_idx]["id"],
            "vacancy_name": df_vac_base.iloc[vac_idx]["vacancy_name"],
            "baseline_score": score,
            "rank": rank + 1
        })

baseline_result = pd.DataFrame(results)

In [87]:
baseline_result.head()

,resume_id,resume_title,vacancy_id,vacancy_name,baseline_score,rank
0,27934,Data Scientist,3155,Data scientist (ML),0.147904,1
1,27934,Data Scientist,13595,Data Analyst в управление комплаенс,0.126044,2
2,27934,Data Scientist,12776,Senior Data Scientist,0.125925,3
3,27934,Data Scientist,11068,Инженер данных (Data Engineer),0.118100,4
4,27934,Data Scientist,14846,Аналитик данных / Ad-hoc-аналитик,0.113316,5


### Проверка на метриках

#### Skills Overlap@5

Метрика показывает, насколько хорошо baseline в top-5 подбирает вакансии, у которых навыки пересекаются с навыками резюме

In [88]:
def skills_overlap_ratio(resume_skills, vacancy_skills):
    if not isinstance(resume_skills, list):
        resume_skills = []
    if not isinstance(vacancy_skills, list):
        vacancy_skills = []

    resume_set = set(str(skill).strip().lower() for skill in resume_skills if pd.notna(skill))
    vacancy_set = set(str(skill).strip().lower() for skill in vacancy_skills if pd.notna(skill))

    if len(vacancy_set) == 0:
        return 0.0

    return len(resume_set & vacancy_set) / len(vacancy_set)

resume_skills_df = df_rez_base[["id", "skills_list"]].rename(columns={
    "id": "resume_id",
    "skills_list": "resume_skills"
})

vacancy_skills_df = df_vac_base[["id", "skills_list"]].rename(columns={
    "id": "vacancy_id",
    "skills_list": "vacancy_skills"
})

baseline_eval = (
    baseline_result
    .merge(resume_skills_df, on="resume_id", how="left")
    .merge(vacancy_skills_df, on="vacancy_id", how="left")
)

baseline_eval["skills_overlap"] = baseline_eval.apply(
    lambda row: skills_overlap_ratio(row["resume_skills"], row["vacancy_skills"]),
    axis=1
)

skills_overlap_5 = (
    baseline_eval
    .sort_values(["resume_id", "baseline_score"], ascending=[True, False])
    .groupby("resume_id")
    .head(5)
    .groupby("resume_id")["skills_overlap"]
    .mean()
    .reset_index(name="skills_overlap@5")
)

skills_overlap_summary = (
    skills_overlap_5
    .drop(columns="resume_id")
    .agg(["mean", "median", "std", "min", "max"])
    .T
)

display(skills_overlap_5.head())
display(skills_overlap_summary)

,resume_id,skills_overlap@5
0,2,0.172683
1,3,0.164698
2,4,0.056190
3,6,0.548578
4,8,0.373810


,mean,median,std,min,max
skills_overlap@5,0.230446,0.218889,0.178276,0.0,0.905556


#### Intersection Count@5

Метрика показывает, сколько вакансий из top-5 baseline пересеклись с полуэталоном

In [89]:
pseudo_top = result.copy()

baseline_top5 = (
    baseline_result
    .sort_values(["resume_id", "rank"], ascending=[True, True])
    .groupby("resume_id")
    .head(5)
)

pseudo_lists = pseudo_top.groupby("resume_id")["vacancy_id"].apply(set)
baseline_lists = baseline_top5.groupby("resume_id")["vacancy_id"].apply(set)

comparison = pd.concat([pseudo_lists, baseline_lists], axis=1).dropna()
comparison.columns = ["pseudo_vacancies", "baseline_vacancies"]

comparison["intersection_count@5"] = comparison.apply(
    lambda row: len(row["pseudo_vacancies"] & row["baseline_vacancies"]),
    axis=1
)

intersection_summary = (
    comparison["intersection_count@5"]
    .agg(["mean", "median", "std", "min", "max"])
    .to_frame(name="intersection_count@5")
)

display(comparison.head())
display(intersection_summary)

,pseudo_vacancies,baseline_vacancies,intersection_count@5
resume_id,,,
2,"{4195, 8164, 9445, 4486, 6507, 4205, 5136, 110...","{7175, 14708, 6645, 6454, 15447}",0
3,"{12163, 2693, 7174, 2311, 4486, 4363, 654, 513...","{13415, 10505, 10348, 11990, 15447}",0
4,"{736, 7042, 10404, 7174, 6183, 14664, 4363, 16...","{13671, 3629, 10100, 10165, 7959}",0
6,"{576, 2819, 10404, 12901, 4486, 12676, 9307, 1...","{8164, 14708, 6645, 6454, 14846}",0
8,"{12163, 7428, 11523, 4486, 2311, 10129, 14099,...","{6470, 266, 5999, 8475, 13595}",1


,intersection_count@5
mean,0.633933
median,0.000000
std,1.131614
min,0.000000
max,5.000000


#### Mean Top-5 Score

Средний baseline_score по top-5.

In [90]:
top5 = (
    baseline_result
    .sort_values(["resume_id", "baseline_score"], ascending=[True, False])
    .groupby("resume_id")
    .head(5)
)

mean_top5_score = (
    top5
    .groupby("resume_id")["baseline_score"]
    .mean()
    .reset_index(name="mean_top5_score")
)

mean_top5_summary = (
    mean_top5_score["mean_top5_score"]
    .agg(["mean", "median", "std", "min", "max"])
    .to_frame(name="mean_top5_score")
)

display(mean_top5_score.head())
display(mean_top5_summary)

,resume_id,mean_top5_score
0,2,0.136121
1,3,0.146077
2,4,0.109663
3,6,0.260188
4,8,0.141111


,mean_top5_score
mean,0.163351
median,0.136942
std,0.094343
min,0.033128
max,0.633860


#### Intra-list Diversity@5

Показывает, насколько вакансии внутри одного top-5 похожи друг на друга

In [91]:
def jaccard_similarity(skills_a, skills_b):
    if not isinstance(skills_a, list):
        skills_a = []
    if not isinstance(skills_b, list):
        skills_b = []

    set_a = set(str(skill).strip().lower() for skill in skills_a if pd.notna(skill))
    set_b = set(str(skill).strip().lower() for skill in skills_b if pd.notna(skill))

    union = set_a | set_b
    intersection = set_a & set_b

    if len(union) == 0:
        return 0.0

    return len(intersection) / len(union)

vacancy_skills_df = df_vac_base[["id", "skills_list"]].rename(columns={
    "id": "vacancy_id",
    "skills_list": "vacancy_skills"
})

baseline_eval = baseline_result.merge(
    vacancy_skills_df,
    on="vacancy_id",
    how="left"
)

top5 = (
    baseline_eval
    .sort_values(["resume_id", "baseline_score"], ascending=[True, False])
    .groupby("resume_id")
    .head(5)
)

rows = []

for resume_id, group in top5.groupby("resume_id"):
    vacancy_skills_lists = group["vacancy_skills"].tolist()

    pair_diversities = []

    for skills_a, skills_b in combinations(vacancy_skills_lists, 2):
        sim = jaccard_similarity(skills_a, skills_b)
        div = 1 - sim
        pair_diversities.append(div)

    if len(pair_diversities) == 0:
        diversity = 0.0
    else:
        diversity = sum(pair_diversities) / len(pair_diversities)

    rows.append({
        "resume_id": resume_id,
        "intra_list_diversity@5": diversity
    })

intra_list_diversity_5 = pd.DataFrame(rows)

intra_list_diversity_summary = (
    intra_list_diversity_5["intra_list_diversity@5"]
    .agg(["mean", "median", "std", "min", "max"])
    .to_frame(name="intra_list_diversity@5")
)

display(intra_list_diversity_5.head())
display(intra_list_diversity_summary)

,resume_id,intra_list_diversity@5
0,2,0.843273
1,3,0.861111
2,4,0.945846
3,6,0.854728
4,8,0.916111


,intra_list_diversity@5
mean,0.873128
median,0.892933
std,0.090014
min,0.000000
max,0.997674


## Вывод

- Были обработаны сырые тексты парсинга карточек резюме и вакансий с HeadHunter
- В результате были получены 11782 резюме и 571 вакансия (при первичной обработке, при фильтрации по указанным навыкам)
- был построен "псевдоэталон" - расширенное множество потенциально релевантных вакансий, основанных на пересечении навыков (в большей степени) и соответствии опыта (в меньшей)
- На подходе TF-IDF + cosine similarity по полным текстам был построен baseline 
- Оценка качества показала, что данный baseline находит только близкие вакансии с ограниченным качеством - стоит использовать более продвинутые методы сопоставления


# Бейслайн попытка 2